# 05 — Export and reproducibility
This notebook asks: *how can a researcher export calculated tables and record enough provenance to identify the analysis?* It independently creates Capacity results from the public synthetic source and exports supported CSV and Parquet files. Units remain those declared by each table, and positive current denotes charge. Temporary exports are local, contain no private data, and are removed with the notebook process.


In [ ]:
import tempfile
from pathlib import Path


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "data" / "synthetic_cycling.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the gregorithm repository folder.")


PROJECT_ROOT = find_project_root()
LOCAL_FILE = None  # Optional: Path(r"D:/your-local/canonical-file.csv")
SOURCE = (
    Path(LOCAL_FILE)
    if LOCAL_FILE is not None
    else PROJECT_ROOT / "examples" / "data" / "synthetic_cycling.csv"
)
_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Input: {SOURCE.name}")
print(f"Temporary result area: {WORK}")

In [ ]:
from gregorithm import ElaborationRequest, ImportRequest, elaborate, import_source

imported = import_source(
    ImportRequest(SOURCE, WORK / "import", "synthetic-cell", "import-job", "import-v1")
)
elaborated = elaborate(
    ElaborationRequest(
        imported.ref.path / "points" / "canonical.parquet",
        imported.report.content_hash,
        WORK / "elaboration",
        "synthetic-cell",
        "elaboration-job",
        "elaboration-v1",
    )
)
print(f"Retained rows: {imported.report.retained_rows}")
print(f"Cycles: {elaborated.cycles}; half-cycles: {elaborated.half_cycles}")

In [ ]:
from gregorithm import CapacityParameters, CapacityRequest, analyze_capacity

capacity_parameters = CapacityParameters(reference_cycle=1)
capacity_result = analyze_capacity(
    CapacityRequest(
        elaborated.ref.path,
        elaborated.fingerprint,
        WORK / "capacity",
        "synthetic-cell",
        "capacity-job",
        "capacity-v1",
        capacity_parameters,
    )
)

In [ ]:
from gregorithm import ExportFormat, ExportRequest, export_table

csv_path = WORK / "capacity.csv"
parquet_path = WORK / "elaborated_points.parquet"
capacity_receipt = export_table(
    ExportRequest(capacity_result.ref.path, "tables/capacity.parquet", csv_path, ExportFormat.CSV)
)
points_receipt = export_table(
    ExportRequest(
        elaborated.ref.path,
        "points/elaborated.parquet",
        parquet_path,
        ExportFormat.PARQUET,
        columns=("time_s", "current_a", "voltage_v", "cycle", "half_cycle"),
    )
)
print(capacity_receipt)
print(points_receipt)

In [ ]:
print(csv_path.read_text(encoding="utf-8"))

In [ ]:
import json

import gregorithm

elaboration_manifest = json.loads(
    (elaborated.ref.path / "manifest.json").read_text(encoding="utf-8")
)
capacity_manifest = json.loads(
    (capacity_result.ref.path / "manifest.json").read_text(encoding="utf-8")
)
reproducibility_record = {
    "gregorithm_version": gregorithm.__version__,
    "canonical_content_hash": imported.report.content_hash,
    "elaboration_fingerprint": elaborated.fingerprint,
    "capacity_fingerprint": capacity_result.fingerprint,
    "capacity_parameters": capacity_parameters.as_dict(),
    "export_sha256": capacity_receipt.sha256,
    "elaboration_operation": elaboration_manifest["operation"],
    "capacity_operation": capacity_manifest["operation"],
}
reproducibility_record

CSV and Parquet exports contain selected numerical tables without hidden provenance sidecars. Keep a separate record of software version, canonical content hash, result fingerprints, parameters, diagnostics, units, and export checksum. A filename alone is not scientific identity, and displayed rounding must not replace full exported values.
